# Modeling Notebook

This notebook **demonstrates** the ML concepts interactively, complementing `src/train.py`.

**Sections:**
1. Bias-Variance Tradeoff visualisation
2. Effect of regularisation (C) on Logistic Regression
3. Random Forest: OOB error vs n_estimators
4. XGBoost learning curves (train vs val log-loss)
5. MLP training history
6. Final model comparison table
7. Threshold tuning for security needs

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import validation_curve, learning_curve
from sklearn.metrics import f1_score, precision_recall_curve

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

ROOT  = Path('..')
PROC  = ROOT / 'data' / 'processed'
MDIR  = ROOT / 'saved_models'

In [ ]:
# Load preprocessed data
pipeline = joblib.load(MDIR / 'feature_pipeline.pkl')
le       = joblib.load(MDIR / 'label_encoder.pkl')
feature_names = joblib.load(MDIR / 'feature_names.pkl')
class_names   = joblib.load(MDIR / 'class_names.pkl')

from src.preprocessing import load_splits
X_train, y_train_raw, X_val, y_val_raw, X_test, y_test_raw, _ = load_splits(PROC)

X_train_t = pipeline.transform(X_train)
X_val_t   = pipeline.transform(X_val)
X_test_t  = pipeline.transform(X_test)

y_train = le.transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(y_test_raw)

print(f'Train: {X_train_t.shape}, Val: {X_val_t.shape}, Test: {X_test_t.shape}')
print(f'Classes: {class_names}')

## 1. Effect of Regularisation on Logistic Regression

**C = 1/λ** — as C decreases, regularisation increases.

Small C → high bias (underfitting) but low variance.  
Large C → low bias but high variance (overfitting risk).

In [ ]:
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
train_scores, val_scores = [], []

for C in C_values:
    lr = LogisticRegression(C=C, solver='saga', max_iter=500,
                            multi_class='multinomial',
                            class_weight='balanced', random_state=42)
    lr.fit(X_train_t, y_train)
    train_scores.append(f1_score(y_train, lr.predict(X_train_t), average='macro'))
    val_scores.append(f1_score(y_val, lr.predict(X_val_t), average='macro'))

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(C_values, train_scores, 'o-', label='Train Macro-F1')
ax.semilogx(C_values, val_scores,   's--', label='Val Macro-F1')
ax.set_xlabel('C (inverse regularisation strength)', fontsize=12)
ax.set_ylabel('Macro-F1', fontsize=12)
ax.set_title('Bias-Variance Tradeoff — Logistic Regression', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/lr_regularisation_curve.png', dpi=150)
plt.show()
print(f'Best C: {C_values[np.argmax(val_scores)]}')

## 2. Learning Curve — diagnosing bias vs variance

- Train score >> Val score → **High variance** (overfitting) → add regularisation / more data
- Both scores low → **High bias** (underfitting) → add features / more complex model
- Converged and close → good generalisation

In [ ]:
from sklearn.model_selection import learning_curve

lr_best = LogisticRegression(C=1.0, solver='saga', max_iter=500,
                              multi_class='multinomial',
                              class_weight='balanced', random_state=42)

train_sizes, train_sc, val_sc = learning_curve(
    lr_best, X_train_t, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    scoring='f1_macro', cv=3, n_jobs=-1, verbose=0
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_sc.mean(axis=1), 'o-', label='Train')
ax.fill_between(train_sizes,
                train_sc.mean(axis=1) - train_sc.std(axis=1),
                train_sc.mean(axis=1) + train_sc.std(axis=1), alpha=0.15)
ax.plot(train_sizes, val_sc.mean(axis=1), 's--', label='Val')
ax.fill_between(train_sizes,
                val_sc.mean(axis=1) - val_sc.std(axis=1),
                val_sc.mean(axis=1) + val_sc.std(axis=1), alpha=0.15)
ax.set_xlabel('Training set size', fontsize=12)
ax.set_ylabel('Macro-F1', fontsize=12)
ax.set_title('Learning Curve — Logistic Regression', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/lr_learning_curve.png', dpi=150)
plt.show()

## 3. Threshold Tuning for Security

Default threshold = 0.5 (argmax of probabilities).  
In security, we may want to LOWER the threshold for attack classes to increase recall  
(catch more attacks at the cost of more false alarms — Precision-Recall tradeoff).

In [ ]:
# Load best model
import os
model = None
for fname in ['lightgbm.pkl', 'xgboost.pkl', 'random_forest.pkl']:
    if (MDIR / fname).exists():
        model = joblib.load(MDIR / fname)
        print(f'Loaded {fname}')
        break

if model is not None:
    probs = model.predict_proba(X_test_t)
    
    # Binary: DDoS vs rest
    ddos_idx = list(class_names).index('DDoS') if 'DDoS' in class_names else 0
    y_bin = (y_test == ddos_idx).astype(int)
    ddos_probs = probs[:, ddos_idx]
    
    precision, recall, thresholds = precision_recall_curve(y_bin, ddos_probs)
    f1_scores_th = 2 * precision * recall / (precision + recall + 1e-8)
    best_th = thresholds[np.argmax(f1_scores_th[:-1])]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(thresholds, precision[:-1], label='Precision')
    axes[0].plot(thresholds, recall[:-1],    label='Recall')
    axes[0].plot(thresholds, f1_scores_th[:-1], label='F1', linestyle='--')
    axes[0].axvline(best_th, color='red', linestyle=':', label=f'Best threshold={best_th:.2f}')
    axes[0].set_xlabel('Decision Threshold')
    axes[0].set_title('Precision / Recall vs Threshold (DDoS)')
    axes[0].legend()
    
    axes[1].plot(recall, precision)
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve (DDoS vs Rest)')
    
    plt.suptitle('Threshold Tuning for DDoS Detection', fontsize=13)
    plt.tight_layout()
    plt.savefig('../reports/threshold_tuning.png', dpi=150)
    plt.show()
    print(f'Optimal threshold: {best_th:.3f}')

## 4. Final Model Comparison

In [ ]:
from pathlib import Path
comparison_path = ROOT / 'reports' / 'model_comparison.csv'
if comparison_path.exists():
    df_cmp = pd.read_csv(comparison_path)
    print(df_cmp.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(df_cmp))
    width = 0.25
    ax.bar([i-width for i in x], df_cmp['Macro-F1'],    width, label='Macro-F1')
    ax.bar([i       for i in x], df_cmp['Weighted-F1'], width, label='Weighted-F1')
    ax.bar([i+width for i in x], df_cmp['Micro-F1'],    width, label='Micro-F1')
    ax.set_xticks(x)
    ax.set_xticklabels(df_cmp['Model'], rotation=20)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Model Comparison — F1 Scores', fontsize=13)
    ax.legend()
    plt.tight_layout()
    plt.savefig('../reports/model_comparison_chart.png', dpi=150)
    plt.show()
else:
    print('Run src/train.py first to generate comparison results.')